# P09: Applications and advanced tips

`numpy` is the engine under `pandas`, `scipy`, and essentially every neural-data
toolbox. Here we re-double our focus on *axes and shapes* as the same line
of code can produce completely different results depending on which axis it runs
over, and it won't tell you when you have chosen the wrong one.

## The modern random API

In [1]:
import numpy as np

# make a generator with an explicit seed
rng = np.random.default_rng(42)     # replaces the old np.random.seed(...) global state

# a (trials x timepoints) array of a recorded signal
data = rng.normal(0, 1, size=(50, 200))
print(data.shape)

(50, 200)


## Z-scoring: which axis?

In [2]:
# z-score each timepoint across trials  ->  axis=0
z = (data - data.mean(axis=0)) / data.std(axis=0, ddof=1)
print("z shape:", z.shape, " grand mean ~", round(float(z.mean()), 3))

z shape: (50, 200)  grand mean ~ -0.0


:::{admonition} Advanced tips: axis is an analysis choice
:class: tip
`data.mean(axis=0)` averages over trials (giving one value per timepoint);
`data.mean(axis=1)` averages over time (one value per trial). Both run 
and return sensible-looking arrays of the *wrong* thing if you picked the wrong
axis. Always check `.shape` before and after, and write a comment stating what
each axis means. Don't let AI assistants guess about the proper axis.
:::

## Baseline subtraction with broadcasting

In [3]:
# subtract each trial's own pre-stimulus baseline (first 50 samples)
baseline = data[:, :50].mean(axis=1, keepdims=True)    # keepdims -> shape (50, 1)
corrected = data - baseline                            # broadcasts across time
print("baseline:", baseline.shape, " corrected:", corrected.shape)

baseline: (50, 1)  corrected: (50, 200)


In [4]:
# without keepdims the baseline is shape (50,), which does NOT line up with (50, 200)
bad_shape = data[:, :50].mean(axis=1)
print("bad_shape:", bad_shape.shape)
# 'data - bad_shape' would raise; 'data - bad_shape[:, None]' is the explicit fix

bad_shape: (50,)


:::{admonition} Advanced tips: broadcasting can silently do the wrong thing
:class: tip
NumPy's broadcasting is powerful and quiet. Mismatched-but-broadcastable shapes,
such as a `(200,)` row vector where you meant a `(50, 1)` column vector, can
subtract the wrong quantity from every element without any error. Use
`keepdims=True`, add explicit axes with `[:, None]`, and assert shapes
(`assert baseline.shape == (50, 1)`) when it matters.
:::